In [ ]:
import pandas as pd
import numpy as np

# 1. Load the dataset
# Update the filename to match the file you uploaded to Colab
file_path = '/content/delivery_data.csv'
df = pd.read_csv(file_path)

# 2. Basic Information
print("--- DATASET SHAPE ---")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}\n")

print("--- MISSING VALUES ---")
missing_data = df.isnull().sum()
print(missing_data[missing_data > 0]) # Only print columns that actually have missing values

# 3. Create the Business Metrics (Delay Ratio)
# We add a small epsilon (0.0001) to OSRM time to avoid Division by Zero errors!
df['delay_ratio'] = df['actual_time'] / (df['osrm_time'] + 0.0001)

# Define a "Delay Flag" (e.g., if actual time is 20% higher than OSRM time)
df['is_delayed'] = (df['delay_ratio'] > 1.20).astype(int)

# 4. Quick Sanity Check on our new columns
print("\n--- DELAY RATIO SUMMARY ---")
print(df[['actual_time', 'osrm_time', 'delay_ratio', 'is_delayed']].describe())

print("\n--- UNIQUE TRIPS ---")
print(f"Total rows: {df.shape[0]}")
print(f"Unique Trips: {df['trip_uuid'].nunique()}")

--- DATASET SHAPE ---
Rows: 144867, Columns: 24

--- MISSING VALUES ---
source_name         293
destination_name    261
dtype: int64

--- DELAY RATIO SUMMARY ---
         actual_time      osrm_time    delay_ratio     is_delayed
count  144867.000000  144867.000000  144867.000000  144867.000000
mean      416.927527     213.868272       2.120101       0.945019
std       598.103621     308.011085       1.715412       0.227945
min         9.000000       6.000000       0.144000       0.000000
25%        51.000000      27.000000       1.604263       1.000000
50%       132.000000      64.000000       1.857134       1.000000
75%       513.000000     257.000000       2.213481       1.000000
max      4532.000000    1686.000000      77.386847       1.000000

--- UNIQUE TRIPS ---
Total rows: 144867
Unique Trips: 14817


In [ ]:
df.head()
df.info()
df.columns
df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 144867 entries, 0 to 144866
Data columns (total 26 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   data                            144867 non-null  object 
 1   trip_creation_time              144867 non-null  object 
 2   route_schedule_uuid             144867 non-null  object 
 3   route_type                      144867 non-null  object 
 4   trip_uuid                       144867 non-null  object 
 5   source_center                   144867 non-null  object 
 6   source_name                     144574 non-null  object 
 7   destination_center              144867 non-null  object 
 8   destination_name                144606 non-null  object 
 9   od_start_time                   144867 non-null  object 
 10  od_end_time                     144867 non-null  object 
 11  start_scan_to_end_scan          144867 non-null  float64
 12  is_cutoff       

(144867, 26)

In [ ]:
df.head()

,data,trip_creation_time,route_schedule_uuid,route_type,trip_uuid,source_center,source_name,destination_center,destination_name,od_start_time,...,actual_time,osrm_time,osrm_distance,factor,segment_actual_time,segment_osrm_time,segment_osrm_distance,segment_factor,delay_ratio,is_delayed
0,training,2018-09-20 02:35:36.476840,thanos::sroute:eb7bfc78-b351-4c0e-a951-fa3d5c3...,Carting,trip-153741093647649320,IND388121AAA,Anand_VUNagar_DC (Gujarat),IND388620AAB,Khambhat_MotvdDPP_D (Gujarat),2018-09-20 03:21:32.418600,...,14.0,11.0,11.9653,1.272727,14.0,11.0,11.9653,1.272727,1.272716,1
1,training,2018-09-20 02:35:36.476840,thanos::sroute:eb7bfc78-b351-4c0e-a951-fa3d5c3...,Carting,trip-153741093647649320,IND388121AAA,Anand_VUNagar_DC (Gujarat),IND388620AAB,Khambhat_MotvdDPP_D (Gujarat),2018-09-20 03:21:32.418600,...,24.0,20.0,21.7243,1.200000,10.0,9.0,9.7590,1.111111,1.199994,0
2,training,2018-09-20 02:35:36.476840,thanos::sroute:eb7bfc78-b351-4c0e-a951-fa3d5c3...,Carting,trip-153741093647649320,IND388121AAA,Anand_VUNagar_DC (Gujarat),IND388620AAB,Khambhat_MotvdDPP_D (Gujarat),2018-09-20 03:21:32.418600,...,40.0,28.0,32.5395,1.428571,16.0,7.0,10.8152,2.285714,1.428566,1
3,training,2018-09-20 02:35:36.476840,thanos::sroute:eb7bfc78-b351-4c0e-a951-fa3d5c3...,Carting,trip-153741093647649320,IND388121AAA,Anand_VUNagar_DC (Gujarat),IND388620AAB,Khambhat_MotvdDPP_D (Gujarat),2018-09-20 03:21:32.418600,...,62.0,40.0,45.5620,1.550000,21.0,12.0,13.0224,1.750000,1.549996,1
4,training,2018-09-20 02:35:36.476840,thanos::sroute:eb7bfc78-b351-4c0e-a951-fa3d5c3...,Carting,trip-153741093647649320,IND388121AAA,Anand_VUNagar_DC (Gujarat),IND388620AAB,Khambhat_MotvdDPP_D (Gujarat),2018-09-20 03:21:32.418600,...,68.0,44.0,54.2181,1.545455,6.0,5.0,3.9153,1.200000,1.545451,1


In [ ]:
# Create a corridor column
df["corridor"] = df["source_center"].astype(str) + " -> " + df["destination_center"].astype(str)

# How many rows per trip?
rows_per_trip = df.groupby("trip_uuid").size().sort_values(ascending=False)

print("Number of unique trips:", df["trip_uuid"].nunique())
print("Average rows per trip:", rows_per_trip.mean())
print("Median rows per trip:", rows_per_trip.median())
print("Max rows for one trip:", rows_per_trip.max())

display(rows_per_trip.head(20))

# How many rows per trip + corridor?
rows_per_trip_corridor = (
    df.groupby(["trip_uuid", "source_center", "destination_center"])
    .size()
    .sort_values(ascending=False)
)

print("\nNumber of unique trip-corridor combinations:", len(rows_per_trip_corridor))
print("Average rows per trip-corridor:", rows_per_trip_corridor.mean())
print("Median rows per trip-corridor:", rows_per_trip_corridor.median())
print("Max rows for one trip-corridor:", rows_per_trip_corridor.max())

display(rows_per_trip_corridor.head(20))

Number of unique trips: 14817
Average rows per trip: 9.777080380643856
Median rows per trip: 5.0
Max rows for one trip: 101


,0
trip_uuid,
trip-153854305492910872,101
trip-153759210483476123,101
trip-153819749763881430,101
trip-153715938946690081,101
trip-153811219535896559,101
trip-153784927255069118,101
trip-153837029526866991,101
trip-153741795740530104,101
trip-153733005409156789,101



Number of unique trip-corridor combinations: 26368
Average rows per trip-corridor: 5.494045813106796
Median rows per trip-corridor: 3.0
Max rows for one trip-corridor: 81


,,,0
trip_uuid,source_center,destination_center,
trip-153755502932196495,IND160002AAC,IND562132AAA,81
trip-153751271053200074,IND000000ACB,IND600056AAB,79
trip-153802876613714747,IND000000ACB,IND600056AAB,79
trip-153681464570847135,IND000000ACB,IND600056AAB,79
trip-153854253003897121,IND000000ACB,IND600056AAB,79
trip-153690920439662353,IND000000ACB,IND600056AAB,79
trip-153828700829921150,IND000000ACB,IND600056AAB,79
trip-153793999089967312,IND000000ACB,IND600056AAB,79
trip-153837097390448401,IND000000ACB,IND600056AAB,79


In [ ]:
print("Route type distribution:")
display(df["route_type"].value_counts())

print("\nRoute type distribution percentage:")
display((df["route_type"].value_counts(normalize=True) * 100).round(2))

Route type distribution:


,count
route_type,
FTL,99660
Carting,45207



Route type distribution percentage:


,proportion
route_type,
FTL,68.79
Carting,31.21


In [ ]:
df["calculated_factor"] = df["actual_time"] / df["osrm_time"]

check_cols = [
    "actual_time",
    "osrm_time",
    "factor",
    "delay_ratio",
    "calculated_factor",
    "is_delayed"
]

display(df[check_cols].head(20))

print("Correlation: factor vs calculated_factor")
print(df["factor"].corr(df["calculated_factor"]))

print("\nCorrelation: delay_ratio vs calculated_factor")
print(df["delay_ratio"].corr(df["calculated_factor"]))

print("\nMax absolute difference between factor and calculated_factor:")
print((df["factor"] - df["calculated_factor"]).abs().max())

print("\nMax absolute difference between delay_ratio and calculated_factor:")
print((df["delay_ratio"] - df["calculated_factor"]).abs().max())

print("\nDelay label mismatch if threshold is 1.2:")
df["delay_label_check"] = (df["delay_ratio"] > 1.2).astype(int)
print((df["is_delayed"] != df["delay_label_check"]).sum())

print("\nDelay class distribution:")
display(df["is_delayed"].value_counts())
display((df["is_delayed"].value_counts(normalize=True) * 100).round(2))

,actual_time,osrm_time,factor,delay_ratio,calculated_factor,is_delayed
0,14.0,11.0,1.272727,1.272716,1.272727,1
1,24.0,20.0,1.200000,1.199994,1.200000,0
2,40.0,28.0,1.428571,1.428566,1.428571,1
3,62.0,40.0,1.550000,1.549996,1.550000,1
4,68.0,44.0,1.545455,1.545451,1.545455,1
5,15.0,11.0,1.363636,1.363624,1.363636,1
6,44.0,17.0,2.588235,2.588220,2.588235,1
7,65.0,29.0,2.241379,2.241372,2.241379,1
8,76.0,39.0,1.948718,1.948713,1.948718,1
9,102.0,45.0,2.266667,2.266662,2.266667,1


Correlation: factor vs calculated_factor
1.0

Correlation: delay_ratio vs calculated_factor
0.9999999999902714

Max absolute difference between factor and calculated_factor:
7.105427357601002e-15

Max absolute difference between delay_ratio and calculated_factor:
0.0006890538868233875

Delay label mismatch if threshold is 1.2:
0

Delay class distribution:


,count
is_delayed,
1,136902
0,7965


,proportion
is_delayed,
1,94.5
0,5.5


In [ ]:
# First, sort so the largest actual_time comes last within each trip-corridor group
df_clean_base = df.dropna(subset=[
    "trip_uuid",
    "source_center",
    "destination_center",
    "actual_time",
    "osrm_time",
    "actual_distance_to_destination",
    "route_type",
    "trip_creation_time"
]).copy()

# Remove impossible rows before final selection
df_clean_base = df_clean_base[
    (df_clean_base["actual_time"] > 0) &
    (df_clean_base["osrm_time"] > 0) &
    (df_clean_base["actual_distance_to_destination"] > 0)
].copy()

# Select final cumulative record per trip-corridor using maximum actual_time
idx = (
    df_clean_base
    .groupby(["trip_uuid", "source_center", "destination_center"])["actual_time"]
    .idxmax()
)

trip_df = df_clean_base.loc[idx].copy()

# Recalculate delay ratio and label after cleaning
trip_df["delay_ratio"] = trip_df["actual_time"] / trip_df["osrm_time"]
trip_df["is_delayed"] = (trip_df["delay_ratio"] > 1.2).astype(int)

# Create corridor ID
trip_df["corridor"] = (
    trip_df["source_center"].astype(str)
    + " -> " +
    trip_df["destination_center"].astype(str)
)

print("Raw data shape:", df.shape)
print("Clean trip-corridor data shape:", trip_df.shape)

print("\nUnique trips in clean data:", trip_df["trip_uuid"].nunique())
print("Unique corridors in clean data:", trip_df["corridor"].nunique())
print("Unique source centers:", trip_df["source_center"].nunique())
print("Unique destination centers:", trip_df["destination_center"].nunique())

print("\nRoute type distribution after cleaning:")
display(trip_df["route_type"].value_counts())
display((trip_df["route_type"].value_counts(normalize=True) * 100).round(2))

print("\nDelay distribution after cleaning:")
display(trip_df["is_delayed"].value_counts())
display((trip_df["is_delayed"].value_counts(normalize=True) * 100).round(2))

display(trip_df.head())

Raw data shape: (144867, 29)
Clean trip-corridor data shape: (26368, 29)

Unique trips in clean data: 14817
Unique corridors in clean data: 2783
Unique source centers: 1508
Unique destination centers: 1481

Route type distribution after cleaning:


,count
route_type,
FTL,13939
Carting,12429


,proportion
route_type,
FTL,52.86
Carting,47.14



Delay distribution after cleaning:


,count
is_delayed,
1,25005
0,1363


,proportion
is_delayed,
1,94.83
0,5.17


,data,trip_creation_time,route_schedule_uuid,route_type,trip_uuid,source_center,source_name,destination_center,destination_name,od_start_time,...,factor,segment_actual_time,segment_osrm_time,segment_osrm_distance,segment_factor,delay_ratio,is_delayed,corridor,calculated_factor,delay_label_check
125019,training,2018-09-12 00:00:16.535741,thanos::sroute:d7c989ba-a29b-4a0b-b2f4-288cdc6...,FTL,trip-153671041653548748,IND209304AAA,Kanpur_Central_H_6 (Uttar Pradesh),IND000000ACB,Gurgaon_Bilaspur_HB (Haryana),2018-09-12 16:39:46.858469,...,2.224924,20.0,10.0,15.0693,2.000000,2.224924,1,IND209304AAA -> IND000000ACB,2.224924,1
125001,training,2018-09-12 00:00:16.535741,thanos::sroute:d7c989ba-a29b-4a0b-b2f4-288cdc6...,FTL,trip-153671041653548748,IND462022AAA,Bhopal_Trnsport_H (Madhya Pradesh),IND209304AAA,Kanpur_Central_H_6 (Uttar Pradesh),2018-09-12 00:00:16.535741,...,2.139175,22.0,3.0,5.3898,7.333333,2.139175,1,IND462022AAA -> IND209304AAA,2.139175,1
66270,training,2018-09-12 00:00:22.886430,thanos::sroute:3a1b0ab2-bb0b-4c53-8c59-eb2a2c0...,Carting,trip-153671042288605164,IND561203AAB,Doddablpur_ChikaDPP_D (Karnataka),IND562101AAA,Chikblapur_ShntiSgr_D (Karnataka),2018-09-12 02:03:09.655591,...,1.807692,15.0,7.0,6.9464,2.142857,1.807692,1,IND561203AAB -> IND562101AAA,1.807692,1
66267,training,2018-09-12 00:00:22.886430,thanos::sroute:3a1b0ab2-bb0b-4c53-8c59-eb2a2c0...,Carting,trip-153671042288605164,IND572101AAA,Tumkur_Veersagr_I (Karnataka),IND561203AAB,Doddablpur_ChikaDPP_D (Karnataka),2018-09-12 00:00:22.886430,...,2.285714,20.0,3.0,3.8074,6.666667,2.285714,1,IND572101AAA -> IND561203AAB,2.285714,1
131427,training,2018-09-12 00:00:33.691250,thanos::sroute:de5e208e-7641-45e6-8100-4d9fb1e...,FTL,trip-153671043369099517,IND000000ACB,Gurgaon_Bilaspur_HB (Haryana),IND160002AAC,Chandigarh_Mehmdpur_H (Punjab),2018-09-14 03:40:17.106733,...,2.882075,275.0,28.0,32.8506,9.821429,2.882075,1,IND000000ACB -> IND160002AAC,2.882075,1


The dataset contains 13,669 rows but only 1,491 unique trips, with an average of 9.17 records per trip. This indicates that the raw data is not one-row-per-trip; instead, it contains repeated cumulative scan/checkpoint records for the same trip and corridor. The same trip-source-destination combination can appear multiple times, with the maximum being 78 records for one trip-corridor.

For trip-level ETA prediction and graph construction, using all raw rows directly would over-count repeated movements and inflate corridor traffic. Therefore, I will create a cleaned modeling table by selecting the final cumulative record for each trip_uuid, source_center, and destination_center combination.

The delay_ratio is verified as actual_time / osrm_time, and the is_delayed label exactly matches the rule delay_ratio > 1.2. Around 94% of raw records are delayed, which means delay classification is highly imbalanced. Therefore, ETA regression will be the primary modeling task, while SLA breach classification should be evaluated carefully using precision/recall rather than accuracy alone.

In [ ]:
# Compare selected max actual_time row with max distance row per trip-corridor

max_time = (
    df_clean_base
    .loc[
        df_clean_base.groupby(["trip_uuid", "source_center", "destination_center"])["actual_time"].idxmax(),
        ["trip_uuid", "source_center", "destination_center", "actual_time", "actual_distance_to_destination"]
    ]
    .rename(columns={
        "actual_time": "max_time_selected_actual_time",
        "actual_distance_to_destination": "distance_at_max_time"
    })
)

max_distance = (
    df_clean_base
    .loc[
        df_clean_base.groupby(["trip_uuid", "source_center", "destination_center"])["actual_distance_to_destination"].idxmax(),
        ["trip_uuid", "source_center", "destination_center", "actual_time", "actual_distance_to_destination"]
    ]
    .rename(columns={
        "actual_time": "time_at_max_distance",
        "actual_distance_to_destination": "max_distance"
    })
)

check_final_row = max_time.merge(
    max_distance,
    on=["trip_uuid", "source_center", "destination_center"],
    how="left"
)

check_final_row["same_time_and_distance_row"] = (
    check_final_row["max_time_selected_actual_time"] == check_final_row["time_at_max_distance"]
)

print("Do max actual_time and max distance usually select the same row?")
display(check_final_row["same_time_and_distance_row"].value_counts())
display((check_final_row["same_time_and_distance_row"].value_counts(normalize=True) * 100).round(2))

display(check_final_row.head(20))

Do max actual_time and max distance usually select the same row?


,count
same_time_and_distance_row,
True,25382
False,986


,proportion
same_time_and_distance_row,
True,96.26
False,3.74


,trip_uuid,source_center,destination_center,max_time_selected_actual_time,distance_at_max_time,time_at_max_distance,max_distance,same_time_and_distance_row
0,trip-153671041653548748,IND209304AAA,IND000000ACB,732.0,383.759164,732.0,383.759164,True
1,trip-153671041653548748,IND462022AAA,IND209304AAA,830.0,440.973689,830.0,440.973689,True
2,trip-153671042288605164,IND561203AAB,IND562101AAA,47.0,24.644021,47.0,24.644021,True
3,trip-153671042288605164,IND572101AAA,IND561203AAB,96.0,48.542890,96.0,48.542890,True
4,trip-153671043369099517,IND000000ACB,IND160002AAC,611.0,237.439610,336.0,242.309306,False
5,trip-153671043369099517,IND562132AAA,IND000000ACB,2736.0,1689.964663,2736.0,1689.964663,True
6,trip-153671046011330457,IND400072AAB,IND401104AAA,59.0,17.175274,59.0,17.175274,True
7,trip-153671052974046625,IND583101AAA,IND583201AAA,147.0,59.530350,147.0,59.530350,True
8,trip-153671052974046625,IND583119AAA,IND583101AAA,131.0,41.317614,131.0,41.317614,True
9,trip-153671052974046625,IND583201AAA,IND583119AAA,63.0,26.600536,63.0,26.600536,True


In [ ]:
df_clean_base = df.dropna(subset=[
    "trip_uuid",
    "source_center",
    "destination_center",
    "actual_time",
    "osrm_time",
    "actual_distance_to_destination",
    "route_type",
    "trip_creation_time"
]).copy()

df_clean_base = df_clean_base[
    (df_clean_base["actual_time"] > 0) &
    (df_clean_base["osrm_time"] > 0) &
    (df_clean_base["actual_distance_to_destination"] > 0)
].copy()

idx = (
    df_clean_base
    .groupby(["trip_uuid", "source_center", "destination_center"])["actual_time"]
    .idxmax()
)

trip_df = df_clean_base.loc[idx].copy()

trip_df["delay_ratio"] = trip_df["actual_time"] / trip_df["osrm_time"]
trip_df["is_delayed"] = (trip_df["delay_ratio"] > 1.2).astype(int)

trip_df["corridor"] = (
    trip_df["source_center"].astype(str)
    + " -> "
    + trip_df["destination_center"].astype(str)
)

print("Clean trip_df shape:", trip_df.shape)
print("Unique trips:", trip_df["trip_uuid"].nunique())
print("Unique corridors:", trip_df["corridor"].nunique())

Clean trip_df shape: (26368, 29)
Unique trips: 14817
Unique corridors: 2783


In [ ]:
print("Raw df shape:", df.shape)

expected_groups = df.groupby(
    ["trip_uuid", "source_center", "destination_center"]
).ngroups

print("Expected trip-corridor rows:", expected_groups)
print("Current trip_df shape:", trip_df.shape)

print("Unique trips in trip_df:", trip_df["trip_uuid"].nunique())
print("Unique corridors in trip_df:", trip_df["corridor"].nunique())

Raw df shape: (144867, 29)
Expected trip-corridor rows: 26368
Current trip_df shape: (26368, 29)
Unique trips in trip_df: 14817
Unique corridors in trip_df: 2783


In [ ]:
import os

os.makedirs("data/processed", exist_ok=True)

trip_df.to_csv("data/processed/trip_corridor_cleaned.csv", index=False)

print("Saved final cleaned trip-corridor dataset.")
print(trip_df.shape)

Saved final cleaned trip-corridor dataset.
(26368, 29)


After reducing the raw scan-level data to one final record per trip-source-destination corridor, the maximum actual_time record matched the maximum distance record in 95.92% of cases. This validates the cleaning assumption that the row with maximum actual_time generally represents the final cumulative state of a trip-corridor movement.

However, the initial top delayed corridors are dominated by corridors with only one trip and extremely high delay ratios. These are useful as anomaly cases, but they should not be directly treated as chronic bottlenecks. For reliable bottleneck analysis, I will apply a minimum trip-count threshold and also consider SLA breach contribution, not just median delay ratio.